In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / 'src').is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.database.db import get_engine
from dotenv import load_dotenv

load_dotenv(RAIZ / '.env')
engine = get_engine()

df = pd.read_sql('SELECT * FROM staging.indicadores_macro ORDER BY data', engine)
df['data'] = pd.to_datetime(df['data'])
df = df.set_index('data')

print(f'{df.shape[0]} meses | {df.index.min():%b/%Y} → {df.index.max():%b/%Y}')
df.tail(3)

In [ ]:
# shift total = lag econômico (bloco 7) + atraso de publicação
feat = pd.DataFrame({
    'ipca':          df['ipca_pct'],                            # alvo
    'd_selic':       df['taxa_selic'].diff().shift(1),          # Copom divulga no mês
    'd_cambio':      df['cambio_usd'].pct_change().shift(4),    # diário, sem atraso
    'd_ibcbr':       df['ibcbr'].pct_change().shift(3),         # 1 econ + 2 publicação
    'd_focus':       df['ipca_focus'].diff().shift(5),          # semanal, quase tempo real
    'desemprego':    df['desemprego'].shift(2),                 # PNAD-C ~1m de atraso
    'mes':           df.index.month,                            # sazonalidade
})
CANDIDATAS = ['d_selic', 'd_cambio', 'd_ibcbr', 'd_focus', 'desemprego']

# alvo = IPCA do MÊS SEGUINTE (previsão de 1 passo à frente)
feat['y'] = feat['ipca'].shift(-1)

feat = feat.dropna(subset=['y'])
print(feat.tail())

In [ ]:
def rolling_origin_naive(feat: pd.DataFrame, meses_teste: int = 24):
    """Prevê o IPCA do mês seguinte usando o último valor conhecido."""
    feat = feat.sort_index().dropna(subset=['y', 'ipca'])

    if len(feat) <= meses_teste:
        raise ValueError("Dados insuficientes para treino e teste.")

    erros = []

    for i in range(len(feat) - meses_teste, len(feat)):
        treino = feat.iloc[:i]
        real = feat['y'].iloc[i]
        previsto = treino['ipca'].iloc[-1]
        erros.append(abs(real - previsto))

    return np.mean(erros)


mae_naive = rolling_origin_naive(feat)
print(f"Naive: MAE = {mae_naive:.3f} p.p.")